In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import current_timestamp, current_date, input_file_name, lit
import datetime

STREAM_NAME   = "device_metrics"
BUCKET        = "s3://energy-pipline-prod"
SOURCE_TABLE  = "energy_catalog.raw.src_device_metrics"
BRONZE_PATH   = f"{BUCKET}/bronze/device_metrics/"
CATALOG_TABLE = "energy_catalog.raw.bronze_device_metrics"
batch_date    = datetime.date.today().strftime("%Y-%m-%d")
print(f"Source : {SOURCE_TABLE}")
print(f"Target : {CATALOG_TABLE}")
print(f"Date   : {batch_date}")

In [0]:
device_schema = StructType([
    StructField("household_id",        StringType(),  True),
    StructField("device_category",     StringType(),  True),
    StructField("device_brand",        StringType(),  True),
    StructField("device_model",        StringType(),  True),
    StructField("maintenance_status",  StringType(),  True),
    StructField("installation_region", StringType(),  True),
    StructField("runtime_hours",       DoubleType(),  True),
    StructField("device_power_kw",     DoubleType(),  True),
    StructField("motor_speed_rpm",     DoubleType(),  True),
    StructField("efficiency_ratio",    DoubleType(),  True),
    StructField("energy_draw_kwh",     DoubleType(),  True),
    StructField("heat_output",         DoubleType(),  True),
    StructField("cooling_load",        DoubleType(),  True),
    StructField("device_voltage",      DoubleType(),  True),
    StructField("device_current",      DoubleType(),  True),
    StructField("device_temperature",  DoubleType(),  True),
])
print(f"Schema ready: {len(device_schema.fields)} columns")

In [0]:
df_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .option("badRecordsPath", BAD_RECORDS)
    .schema(device_schema)
    .load(RAW_PATH)
)
print(f"Rows read: {df_raw.count()}")
df_raw.show(3)

In [0]:
from pyspark.sql.functions import col

df_bronze = (
    df_raw
    .withColumn("_batch_date",   lit(batch_date))
    .withColumn("_ingestion_ts", current_timestamp())
    .withColumn("_source_file",  col("_metadata.file_path"))
    .withColumn("_stream_name",  lit(STREAM_NAME))
)
print(f"Total columns: {len(df_bronze.columns)}")  # expect 20

In [0]:
(df_bronze.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .partitionBy("_batch_date")
    .save(BRONZE_PATH)
)
print("Write complete")
print(f"Rows in Delta: {spark.read.format('delta').load(BRONZE_PATH).count()}")

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG_TABLE}
    USING DELTA
    LOCATION '{BRONZE_PATH}'
""")
spark.sql(f"SELECT COUNT(*) AS total FROM {CATALOG_TABLE}").show()

In [0]:
spark.sql(f"OPTIMIZE {CATALOG_TABLE} ZORDER BY (household_id)")
print("Done. Bronze_device_metrics complete.") 